Import libraries

In [1]:
import numpy as np
import torch
import random
random.seed(4101)
torch.manual_seed(4101)
import pandas as pd

from data_pipeline import build_pipeline, inverse_transform, load_scaler
from vae_module import VAEConfig, train_vae, encode, decode, load_vae

Import data

In [2]:
wti_df = pd.read_csv('../data/wti_prices.csv', index_col=0, parse_dates=['Date'])
wti_df = np.log(wti_df).diff().dropna()
cols = list(wti_df.columns)
print(wti_df.shape)
print(cols)

(480, 1)
['WTI_price']


In [3]:
predictors = pd.read_csv('../data/predictor_data.csv', index_col=0, parse_dates=['Date'])
cols = list(predictors.columns)
print(predictors.shape)
print(cols)

(481, 8)
['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD']


In [4]:
# Combine TB3MS and WTI data by month (index)
combined_df = predictors.join(wti_df, how='inner').sort_index()

# Optional: set a clear index name
combined_df.index.name = "month"

#filter data up to 2014-01-01 for training data
combined_df = combined_df[combined_df.index <= '2014-01-01']

print(combined_df.shape)
print(combined_df.isna().sum())

(336, 9)
CPI          0
TB3MS        0
M1           0
M2           0
AUD_USD      0
CAD_USD      0
NZD_USD      0
ZAR_USD      0
WTI_price    0
dtype: int64


Stationarity

In [5]:
from statsmodels.tsa.stattools import adfuller

# For each column we perform adf test
for column in combined_df.columns:
    result = adfuller(combined_df[column].dropna())
    print(f"ADF Statistic for {column}: {result[0]}")
    print(f"p-value for {column}: {result[1]}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value}")

# We see that for all columns except WTI price it is not stationary, so we take log difference to make it stationary, for every column except WTI price, which is already stationary
for column in combined_df.columns:
    if column != 'WTI_price':
        combined_df[column] = np.log(combined_df[column]).diff().dropna()



ADF Statistic for CPI: -0.349125611730296
p-value for CPI: 0.9182527489186103
Critical Values:
   1%: -3.4507587628808922
   5%: -2.870530068560499
   10%: -2.5715597727381647
ADF Statistic for TB3MS: -1.928922065668171
p-value for TB3MS: 0.3185799385364664
Critical Values:
   1%: -3.450632157720528
   5%: -2.870474482366864
   10%: -2.5715301325443787
ADF Statistic for M1: 3.0977607356548615
p-value for M1: 1.0
Critical Values:
   1%: -3.4503836022181056
   5%: -2.8703653471616826
   10%: -2.571471939191249
ADF Statistic for M2: 3.192138306640184
p-value for M2: 1.0
Critical Values:
   1%: -3.4510167751522642
   5%: -2.87064334231426
   10%: -2.5716201744283174
ADF Statistic for AUD_USD: -1.8801382634400816
p-value for AUD_USD: 0.3414616037894379
Critical Values:
   1%: -3.4502011472639724
   5%: -2.8702852297358983
   10%: -2.5714292194077513
ADF Statistic for CAD_USD: -1.298698392514305
p-value for CAD_USD: 0.6297675900976204
Critical Values:
   1%: -3.450081345901191
   5%: -2.8702

In [6]:
cleaned_df = combined_df.dropna()
cleaned_df.to_csv('../data/cleaned_data_VAETIMEGAN.csv', index=False)

In [7]:
from data_pipeline import build_pipeline, inverse_transform, load_scaler
import random
import pickle
random.seed(4101)

# Build windowed dataset from raw training DataFrame
windows, scaler = build_pipeline(
    df=cleaned_df,            # rows = timesteps, columns = variables
    scaler_save_path="scaler.pkl"
)
# windows shape: (num_windows, window_length, num_variables)

with open("windows.pkl", "wb") as f:
    pickle.dump(windows, f)

[Pipeline] Observations: 335 | Variables: 9
[Pipeline] Columns: ['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD', 'WTI_price']
[Pipeline] Window length: 24 | Stride: 1
[Pipeline] Windows to be created: 312
[Pipeline] Scaler: minmax
------------------------------------------------------------
[Pipeline] Normalisation complete.
[Pipeline] Data range after scaling: [0.0000, 1.0000]
[Pipeline] Windows created: 312
[Pipeline] Output shape: (312, 24, 9)  (num_windows, window_length, num_variables)
[Pipeline] Scaler saved to: scaler.pkl
------------------------------------------------------------
[Pipeline] Pipeline complete. Ready for Module 2 (VAE).


In [8]:
import importlib

import vae_module
from vae_module import VAEConfig, train_vae, encode, decode, load_vae
importlib.reload(vae_module)

config_vae = VAEConfig(
    latent_dim=4,
    hidden_dim=64,
    num_epochs=400,
    kl_warmup_epochs=50,
    kl_weight_max=0.005,
    learning_rate=1e-4,
    verbose_every=25,
    random_seed=4101,
)
model_vae, history = train_vae(
    data=windows,
    config=config_vae,
    save_path="vae_checkpoint.pt"
)
model_vae = load_vae("vae_checkpoint.pt")

# Encode to latent sequences — shape (312, 24, 8)
latent_sequences = encode(windows, model_vae, use_mean=True)
print("Latent sequences shape:", latent_sequences.shape)

[VAE] Device: cuda
[VAE] Input dim: 9 | Latent dim: 4 | Hidden dim: 64
[VAE] Windows: 312 | Window length: 24 | Variables: 9
[VAE] Train windows: 266 | Val windows: 46
[VAE] KL warmup: 50 epochs | Max KL weight: 0.005
------------------------------------------------------------
[VAE] Epoch    1/400 | KL weight: 0.0001 | Train — Total: 0.289295, Recon: 0.289255, KL: 0.400000 | Val   — Total: 0.285911, Recon: 0.285871, KL: 0.400000
[VAE] Epoch   25/400 | KL weight: 0.0025 | Train — Total: 0.070650, Recon: 0.069650, KL: 0.400000 | Val   — Total: 0.077606, Recon: 0.076606, KL: 0.400000
[VAE] Epoch   50/400 | KL weight: 0.0050 | Train — Total: 0.034074, Recon: 0.031580, KL: 0.498692 | Val   — Total: 0.039075, Recon: 0.036514, KL: 0.512349
[VAE] Epoch   75/400 | KL weight: 0.0050 | Train — Total: 0.024807, Recon: 0.021442, KL: 0.673008 | Val   — Total: 0.030205, Recon: 0.026723, KL: 0.696431
[VAE] Epoch  100/400 | KL weight: 0.0050 | Train — Total: 0.020651, Recon: 0.017641, KL: 0.601908 | V

In [9]:
latent = encode(windows, model_vae, use_mean=True)
latent_flat = latent.reshape(-1, latent.shape[-1])
print("Latent mean per dimension:", latent_flat.mean(axis=0).round(4))
print("Latent std per dimension: ", latent_flat.std(axis=0).round(4))
print("Min std:", latent_flat.std(axis=0).min().round(6))

Latent mean per dimension: [-0.0172  0.1397 -0.1186 -0.0395]
Latent std per dimension:  [0.2755 0.1736 0.1482 0.2581]
Min std: 0.14822


In [10]:
# ydata-synthetic TimeGAN expects 2D input: (num_samples, seq_len * n_features)
# OR it can handle (num_samples, seq_len, n_features) depending on version
# Safest approach: pass as (num_windows, seq_len, latent_dim) directly

num_windows, seq_len, latent_dim = latent_sequences.shape
print(f"Windows: {num_windows} | Seq len: {seq_len} | Latent dim: {latent_dim}")

Windows: 312 | Seq len: 24 | Latent dim: 4


In [11]:
from ydata_synthetic.synthesizers.timeseries import TimeSeriesSynthesizer
from ydata_synthetic.synthesizers import ModelParameters, TrainParameters

gan_args = ModelParameters(
    batch_size=128,
    lr=5e-4,
    noise_dim=latent_dim,   # 8 — noise input dimension
    layers_dim=64,
    latent_dim=latent_dim,  # 8 — internal latent, match your features
    gamma=1,
)

train_args = TrainParameters(
    epochs=2000,
    sequence_length=seq_len,       # 24
    number_sequences=latent_dim,   # 8 — number of FEATURES, not windows
)

# 1.4.0 expects a pandas DataFrame, not a numpy array
import pandas as pd
latent_2d = latent_sequences.reshape(-1, latent_dim)
latent_df = pd.DataFrame(
    latent_2d,
    columns=[f"latent_{i}" for i in range(latent_dim)]
)

num_cols = [f"latent_{i}" for i in range(latent_dim)]

synth_gan = TimeSeriesSynthesizer(modelname="timegan", model_parameters=gan_args)
synth_gan.fit(
    latent_df,
    train_args,
    num_cols=num_cols,
    cat_cols=[]        # no categorical columns
)

# Save
synth_gan.save("ydata_timegan_latent.pkl")


A DataProcessor is not available for the TimeGAN.



Joint networks training: 100%|██████████| 2000/2000 [43:54<00:00,  1.32s/it]
c:\Users\Josiah Lee\Documents\GitHub\dse4101-project\.venv\lib\site-packages\tensorflow\python\keras\utils\generic_utils.py:486: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


In [10]:
import joblib
synth_gan = joblib.load("ydata_timegan_latent.pkl")
model_vae = load_vae("vae_checkpoint.pt")

In [11]:
num_windows = 312
np.random.seed(4101)
# Generate
synthetic_latent_raw = synth_gan.sample(num_windows)

# Handle list output from ydata-synthetic 1.4.0
import numpy as np
if isinstance(synthetic_latent_raw, list):
    synthetic_latent_arr = np.array(synthetic_latent_raw)
else:
    synthetic_latent_arr = synthetic_latent_raw.values

print("Raw output shape:", synthetic_latent_arr.shape)

# Reshape depending on what shape came out
# Expected: either (num_windows, seq_len, latent_dim) or (num_windows*seq_len, latent_dim)
if synthetic_latent_arr.ndim == 3:
    synthetic_latent = synthetic_latent_arr  # already (312, 24, 8)
elif synthetic_latent_arr.ndim == 2:
    synthetic_latent = synthetic_latent_arr.reshape(num_windows, seq_len, latent_dim)

print("Synthetic latent shape:", synthetic_latent.shape)

# Decode through VAE
synthetic_normalised = decode(synthetic_latent, model_vae)
print("Synthetic normalised shape:", synthetic_normalised.shape)

synthetic_original = inverse_transform(synthetic_normalised, scaler)
print("Synthetic original shape:", synthetic_original.shape)

Raw output shape: (312, 24, 4)
Synthetic latent shape: (312, 24, 4)
Synthetic normalised shape: (312, 24, 9)
Synthetic original shape: (312, 24, 9)


In [13]:
# Compare real latent distribution vs ydata output
real_latent = encode(windows, model_vae, use_mean=True)
print("Real latent mean:", real_latent.mean(axis=(0,1)).round(4))
print("Real latent std: ", real_latent.std(axis=(0,1)).round(4))
print("Real latent min: ", real_latent.min())
print("Real latent max: ", real_latent.max())

print("\nydata output mean:", synthetic_latent_arr.mean(axis=(0,1)).round(4))
print("ydata output std: ", synthetic_latent_arr.std(axis=(0,1)).round(4))
print("ydata output min: ", synthetic_latent_arr.min())
print("ydata output max: ", synthetic_latent_arr.max())

Real latent mean: [-0.0172  0.1397 -0.1186 -0.0395]
Real latent std:  [0.2755 0.1736 0.1482 0.2581]
Real latent min:  -0.99405473
Real latent max:  1.9690363

ydata output mean: [0.2328 0.7299 0.2189 0.2139]
ydata output std:  [0.0992 0.1074 0.0999 0.1048]
ydata output min:  0.0980722
ydata output max:  0.8522797


In [14]:
from evaluation_module import EvaluationConfig, evaluate_all

config_eval = EvaluationConfig(
    n_lags=12,
    random_seed=4101,
    report_save_path="results/results_vaetimegan.txt",
    json_save_path="results/results_vaetimegan.json",
    figure_save_path="results/figure_vaetimegan.png",
)

results = evaluate_all(
    real=windows,
    synthetic=synthetic_normalised,
    config=config_eval,
    variable_names=["CPI","TB3MS","M1","M2","AUD_USD","CAD_USD","NZD_USD","ZAR_USD","WTI_price"],
)


  EVALUATION SUITE — running all metrics
  Real: 312 windows | Synthetic: 312 windows
  Variables: ['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD', 'WTI_price']

[1/6] Computing statistical moments...
  CPI: real_std=0.0831 | synth_std=0.0131 | std_ratio=0.1578
  TB3MS: real_std=0.0691 | synth_std=0.0022 | std_ratio=0.0320
  M1: real_std=0.0948 | synth_std=0.0097 | std_ratio=0.1025
  M2: real_std=0.1328 | synth_std=0.0042 | std_ratio=0.0318
  AUD_USD: real_std=0.1089 | synth_std=0.0132 | std_ratio=0.1210
  CAD_USD: real_std=0.0921 | synth_std=0.0041 | std_ratio=0.0448
  NZD_USD: real_std=0.1498 | synth_std=0.0102 | std_ratio=0.0684
  ZAR_USD: real_std=0.1073 | synth_std=0.0063 | std_ratio=0.0589
  WTI_price: real_std=0.1137 | synth_std=0.0174 | std_ratio=0.1529

[2/6] Computing ACF comparison...
  Mean ACF MAE: 0.1769
  CPI: ACF MAE = 0.1740
  TB3MS: ACF MAE = 0.1570
  M1: ACF MAE = 0.1826
  M2: ACF MAE = 0.1319
  AUD_USD: ACF MAE = 0.1911
  CAD_USD: ACF MAE = 

In [15]:
num_windows = 20
np.random.seed(4101)
# Generate
synthetic_latent_raw = synth_gan.sample(num_windows)

# Handle list output from ydata-synthetic 1.4.0
import numpy as np
if isinstance(synthetic_latent_raw, list):
    synthetic_latent_arr = np.array(synthetic_latent_raw)
else:
    synthetic_latent_arr = synthetic_latent_raw.values

print("Raw output shape:", synthetic_latent_arr.shape)

# Reshape depending on what shape came out
# Expected: either (num_windows, seq_len, latent_dim) or (num_windows*seq_len, latent_dim)
if synthetic_latent_arr.ndim == 3:
    synthetic_latent = synthetic_latent_arr  # already (312, 24, 8)
elif synthetic_latent_arr.ndim == 2:
    synthetic_latent = synthetic_latent_arr.reshape(num_windows, seq_len, latent_dim)

print("Synthetic latent shape:", synthetic_latent.shape)

# Decode through VAE
synthetic_normalised = decode(synthetic_latent, model_vae)
print("Synthetic normalised shape:", synthetic_normalised.shape)

synthetic_original = inverse_transform(synthetic_normalised, scaler)
print("Synthetic original shape:", synthetic_original.shape)

synthetic_flat = synthetic_original.reshape(-1, synthetic_original.shape[-1])
print(synthetic_flat.shape)  # (480, 8)
columns = ["CPI","TB3MS","M1","M2","AUD_USD","CAD_USD","NZD_USD","ZAR_USD","WTI_price"]
synthetic_df = pd.DataFrame(synthetic_flat, columns = cleaned_df.columns)
print(synthetic_df.head())

Raw output shape: (20, 24, 4)
Synthetic latent shape: (20, 24, 4)
Synthetic normalised shape: (20, 24, 9)
Synthetic original shape: (20, 24, 9)
(480, 9)
        CPI     TB3MS        M1        M2   AUD_USD   CAD_USD   NZD_USD  \
0  0.002967 -0.078071  0.005822  0.004485  0.004924  0.001208  0.002482   
1  0.002889 -0.077729  0.005802  0.004549  0.004720  0.001616  0.002123   
2  0.002982 -0.078367  0.005813  0.004466  0.004953  0.001123  0.002562   
3  0.002480 -0.075877  0.004715  0.004620  0.002206  0.001954  0.000250   
4  0.002322 -0.072973  0.004281  0.004570  0.000722  0.001384  0.000036   

    ZAR_USD  WTI_price  
0  0.011753   0.006789  
1  0.012134   0.005398  
2  0.011639   0.007065  
3  0.014457  -0.007577  
4  0.014622  -0.012798  
